In [2]:
from airports.airportLookup import AirportLookup
airport_lookup = AirportLookup()
import polars as pl
def add_airport_coords(df: pl.DataFrame, prefix: str) -> pl.DataFrame:
    airport_col = f"{prefix}_airport_ident"
    lat_col = f"{prefix}_airport_lat"
    lon_col = f"{prefix}_airport_lon"

    def get_coords(airport_ident: str | None) -> dict[str, float | None]:
        coords = airport_lookup.get_airport_coordinates(airport_ident)

        if coords is None:
            return {lat_col: None, lon_col: None}

        lat, lon = coords
        return {lat_col: lat, lon_col: lon}

    return (
        df.with_columns(
            pl.col(airport_col)
            .map_elements(
                get_coords,
                return_dtype=pl.Struct({
                    lat_col: pl.Float64,
                    lon_col: pl.Float64,
                }),
            )
            .alias("_coords")
        )
        .unnest("_coords")
    )


In [3]:
from data_engineering.flights.flight_type import flights_df_to_flights, get_flights
from datetime import date
df = get_flights(date(2025,5, 20), date(2025,5,21))
df = df.filter(pl.col("ladd") | pl.col("pia"))
df = add_airport_coords(df, "takeoff")
df = add_airport_coords(df, "landing")

In [4]:
from utils import haversine
count = 0
def process_icao_group(group):
    flights = flights_df_to_flights(group)
    previous_flight = None
    for flight in flights:
        if previous_flight is not None:
            if previous_flight.landing_airport_ident != flight.takeoff_airport_ident:
                distance = haversine(
                    previous_flight.landing_airport_lat,
                    previous_flight.landing_airport_lon,
                    flight.takeoff_airport_lat,
                    flight.takeoff_airport_lon,
                )
                global count
                count +=1
                if distance > 100: # cranking the distance is good.
                    pass
                else:
                    print(airport_lookup.get_Airport_from_airport_ident(flight.takeoff_airport_ident).type)
                    pass
        previous_flight = flight

In [ ]:
for icao_key, group in df.group_by("icao"):
    process_icao_group(group)

In [ ]:
len(flights)

9736

In [ ]:
# use callsign and then check if there's a big jump then there's really a problem
# !!!! This is how I detemrine which icaos are using pia which ones have a jump to not. I can probably elmianate tons of icaos from before the solve. 